# 226. Invert Binary Tree
**Difficulty:** 🟢 Easy · **Topic:** Tree · **LeetCode:** https://leetcode.com/problems/invert-binary-tree/

## 💡 Concepts

**Core concept(s):** Visit every node and **swap its two children** (DFS or BFS).

**Why it applies here:** A mirror image swaps left and right everywhere. Do that swap at every node and the whole tree flips.

**Key intuition:** At each node, swap left and right; then do the same inside both children.

---

### 📚 What is a Binary Tree?
A **binary tree** is nodes in a branching shape: each node holds a value and up to two children (**left**, **right**). The top is the **root**; childless nodes are **leaves**; **height** is the longest root-to-leaf path.
- **In Python:** a small `TreeNode` class with `.val`, `.left`, `.right`.

### 📚 What is DFS (Depth-First Search) / Recursion?
**DFS** dives down one branch as far as possible, then backtracks. It's usually written with **recursion** — a function that calls itself on each child.
- **Complexity:** visits each node once → **O(n)** time; uses call-stack space up to the tree's **height**.

### 📚 What is BFS (Breadth-First Search) / a Queue?
**BFS** explores level by level using a **queue** (a first-in-first-out line): take a node, push its children to the back, repeat.
- **Complexity:** **O(n)** time; the queue can hold up to one full level.
- **In Python:** `collections.deque` (`append` to add, `popleft` to take from the front).

---

**Prerequisite knowledge:**
- Recursion or a queue; tuple-swap in Python.

## 📝 Problem

Flip the tree into its mirror image (swap left/right at every node).

**Example**
```
   4              4
  / \            / \
 2   7   ->     7   2
/ \ / \        / \ / \
1 3 6 9        9 6 3 1
```

> Two approaches, both `O(n)`: recursion vs a queue.

In [ ]:
from typing import Optional, List
from collections import deque

class TreeNode:
    """A single node of a binary tree: a value plus links to up to two children."""
    def __init__(self, val=0, left=None, right=None):
        self.val = val                     # the number stored at this node
        self.left = left                   # the left child (or None)
        self.right = right                 # the right child (or None)

def build_tree(values):
    """Build a tree from a level-order list, LeetCode style (None = missing child)."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0]); q = deque([root]); i = 1
    while q and i < len(values):
        node = q.popleft()                 # the parent we're attaching children to
        if i < len(values):                # attach the left child (if present)
            if values[i] is not None:
                node.left = TreeNode(values[i]); q.append(node.left)
            i += 1
        if i < len(values):                # attach the right child (if present)
            if values[i] is not None:
                node.right = TreeNode(values[i]); q.append(node.right)
            i += 1
    return root

def build_balanced(n):
    """Balanced BST holding 1..n (height ~log n) — used by the benchmark."""
    def helper(lo, hi):
        if lo > hi:
            return None
        mid = (lo + hi) // 2               # middle value becomes the subtree's root
        node = TreeNode(mid)
        node.left = helper(lo, mid - 1)    # smaller values go left
        node.right = helper(mid + 1, hi)   # larger values go right
        return node
    return helper(1, n)

def preorder(root):
    """Collect values in preorder: node, then left, then right."""
    out = []
    def go(n):
        if not n: return
        out.append(n.val); go(n.left); go(n.right)
    go(root); return out

def inorder(root):
    """Collect values in inorder: left, then node, then right (sorted for a BST)."""
    out = []
    def go(n):
        if not n: return
        go(n.left); out.append(n.val); go(n.right)
    go(root); return out

def same_shape(a, b):
    """True if two trees have identical shape and values."""
    if not a and not b: return True        # both empty -> match
    if not a or not b or a.val != b.val: return False  # one empty, or values differ
    return same_shape(a.left, b.left) and same_shape(a.right, b.right)

### Approach 1 — Recursion

**Idea:** Swap the two children, then invert each child.

**Time:** `O(n)`. **Space:** `O(h)`.

In [ ]:
def invert_rec(root: Optional[TreeNode]) -> Optional[TreeNode]:
    if not root:
        return None                        # nothing to flip
    # Swap the two children (each recursively inverted first).
    root.left, root.right = invert_rec(root.right), invert_rec(root.left)
    return root

### Approach 2 — BFS with a Queue

**Idea:** Visit every node with a queue; swap its children as you go.

**Time:** `O(n)`. **Space:** `O(n)`.

In [ ]:
def invert_bfs(root: Optional[TreeNode]) -> Optional[TreeNode]:
    if not root:
        return None
    q = deque([root])
    while q:
        node = q.popleft()
        node.left, node.right = node.right, node.left   # swap this node's children
        if node.left:  q.append(node.left)              # continue to the children
        if node.right: q.append(node.right)
    return root

In [ ]:
# Correctness check (invert, then compare to the expected mirror)
tests = [([4,2,7,1,3,6,9],[4,7,2,9,6,3,1]), ([2,1,3],[2,3,1]), ([],[])]
for vals, exp in tests:
    r1 = invert_rec(build_tree(vals))
    r2 = invert_bfs(build_tree(vals))
    goal = build_tree(exp)
    print(f"{vals} -> inverted to {preorder(r1)}")
    assert same_shape(r1, goal) and same_shape(r2, goal), "mismatch!"
print("\nAll tests passed")

## ⏱️ Empirically Checking the Complexities

Big-O can't be read off a function directly, but it can be **measured**. We time each approach on trees of growing size `n` and read the **doubling ratio** — how much runtime grows when `n` doubles.

| Theoretical | Ratio when `n` → `2n` |
|-------------|-----------------------|
| `O(log n)`    | ≈ **1×** |
| `O(n)`        | ≈ **2×** |
| `O(n²)`       | ≈ **4×** |

We use **balanced** trees (height ~log n) so deep recursion stays safe while every node is still visited.

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark

def make_worst_case(n):
    return (build_balanced(n),)   # fresh tree each call (inverting mutates it)
solutions = {
    "recursion O(n)": invert_rec,
    "bfs       O(n)": invert_bfs,
}
sizes = [1000, 2000, 4000, 8000]

benchmark(solutions, make_worst_case, sizes, plot=True)


## 🧩 Patterns Learned

- **Visit-and-modify:** some tree tasks are just "do a small local change at every node" — any full traversal works.
- **Signal:** "mirror / flip / invert a tree".
- **Related problems:** Symmetric Tree, Same Tree, Binary Tree Level Order.
- **Common pitfalls:** (1) swapping after recursing on already-swapped children (do the swap with a simultaneous assignment); (2) forgetting to return the root.